# 10 Tests for Code, Data, and Model Logic

After refactoring, we can test important project behavior without running full notebooks. Tests are a safety net: they make sure that future changes do not silently break cleaning rules, feature definitions, evaluation assumptions, or prediction behavior.

## 1. What We Test First

The first test suite is intentionally small. It focuses on behavior that is important for the Bank Marketing project and relatively easy to break:

- data-cleaning rules for the target variables `y` and `y_binary`;
- removal of the leakage feature `duration`;
- deterministic feature engineering for `PreviouslyContacted`, `HasAnyLoan`, and `BalanceGroup`;
- calculation of the core classification metrics;
- prediction-threshold behavior;
- availability of the command-line pipeline interface.

We distinguish between two test types:

- **Unit tests** check one function or one small piece of logic in isolation. They should be fast, focused, and easy to understand.
- **Integration tests** check whether multiple components work together. They are usually broader and may take slightly longer. In this project, the first integration test checks that the pipeline CLI exposes the expected commands.

These tests are not an exhaustive production test suite. They represent the first practical layer of automated checks for the local MLOps prototype.


## 2. Test Files and Fixtures

The tests live in `tests/`:

```text
tests/
  conftest.py
  test_dataset.py
  test_features.py
  test_evaluation.py
  test_predict.py
  test_pipeline_cli.py
```

`conftest.py` contains shared fixtures. A fixture is reusable test setup. In this project, `minimal_bank_marketing_input` provides a small Bank Marketing input sample that can be reused across several tests.

The tests use small artificial data examples instead of the full Bank Marketing dataset whenever possible. This keeps the test suite fast and makes the expected behavior easier to inspect.

## 3. Test Naming Convention

The test function names are intentionally descriptive. With `pytest`, long names are common because the test name appears directly in the terminal output when a test fails. A good test name should make the expected behavior visible without opening the test file first.

We use this simple pattern:

```text
test_<function_or_component>_<expected_behavior>
```

Example:

```python
def test_create_business_features_adds_expected_columns():
    ...
```

Read as a sentence, this means that `create_business_features` should add the expected columns. If the test fails, the terminal output already indicates which behavior changed.

Short names such as `test_features()` are easier to type, but they are less useful when debugging. The goal is not to make test names as short as possible, but to make the tested behavior explicit.

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tests").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sorted(path.name for path in (PROJECT_ROOT / "tests").glob("test_*.py"))

['test_dataset.py',
 'test_evaluation.py',
 'test_features.py',
 'test_monitoring.py',
 'test_pipeline_cli.py',
 'test_predict.py',
 'test_serving_api.py']

## 4. Run the Tests

From the repository root, all tests can be run with:

```bash
python -m pytest
```

Only unit tests:

```bash
python -m pytest -m unit
```

Only integration tests:

```bash
python -m pytest -m integration
```

The same commands can later be used in GitHub Actions.

In [7]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Tests failed")

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.1.1, pluggy-1.6.0
rootdir: /home/patri/master/ads2-bank-marketing-project
configfile: pyproject.toml
testpaths: tests
plugins: anyio-4.14.0
collected 14 items

tests/test_dataset.py ..                                                 [ 14%]
tests/test_evaluation.py .                                               [ 21%]
tests/test_features.py ..                                                [ 35%]
tests/test_monitoring.py ..                                              [ 50%]
tests/test_pipeline_cli.py .                                             [ 57%]
tests/test_predict.py .                                                  [ 64%]
tests/test_serving_api.py .....                                          [100%]

=============================== warnings summary ===============================
../../.venvs/ads_II/lib/python3.12/site-packages/fastapi/testclient.py:1
  

## 4. What These Tests Do Not Cover Yet

This first test suite validates key preprocessing, feature engineering, evaluation, prediction, monitoring, CLI, and API behavior. However, it does not yet fully test model quality, data drift, deployment infrastructure, or every possible malformed request.

Useful next additions are:

- stricter data schema tests for required columns, data types, ranges, and allowed categorical values
- model behavior tests, for example minimum expected F1 or ROC-AUC on a fixed validation dataset
- end-to-end pipeline smoke tests using a small fixture dataset
- additional API validation tests for missing fields, invalid values, and unexpected categories
- monitoring and drift tests comparing production inputs with the training distribution
- automated CI execution on every pull request

The testing strategy should remain incremental: start with fast unit tests for stable assumptions, then add integration and end-to-end tests where operational risk increases.